# 24 — MCP Lifecycle and JSON-RPC, From First Principles

Welcome to **Part 3** of this course. Parts 1 (notebooks 01-15) and 2
(notebooks 16-23) taught you to *use* the Azure MCP Server and to *govern*
MCP servers through Azure API Management. Neither required you to know what
actually crosses the wire between an MCP client and server.

Part 3 has **no Azure dependency at all**. You don't need `az login`,
Node.js, or a subscription for notebooks 24-28 -- just the `mcp` Python
package already in `requirements.txt`. The goal is to understand the
protocol well enough to read any MCP server's source code, or build your
own from scratch (notebook 25).

What's ahead:

- **24 (this notebook):** JSON-RPC 2.0 message anatomy and the three-phase
  connection lifecycle.
- **25:** Build a minimal MCP server from scratch with the official SDK.
- **26:** The two primitives the Azure MCP Server barely uses: resources
  and prompts.
- **27:** The two primitives that flip direction: sampling and roots.
- **28:** Transports and capability negotiation, tying Parts 1-3 together.

## JSON-RPC 2.0: three message shapes

MCP's wire format is [JSON-RPC 2.0](https://www.jsonrpc.org/specification).
Every message is exactly one of:

| Shape | Has `id`? | Expects a response? |
| --- | --- | --- |
| **Request** | Yes | Yes -- the other side must reply |
| **Response** | Yes (matches the request) | N/A -- it *is* the reply |
| **Notification** | No | No -- fire-and-forget |

Here's the real `initialize` request/response pair from the
[MCP lifecycle spec](https://modelcontextprotocol.io/specification/2025-06-18/basic/lifecycle):

```json
// Request (client -> server)
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-06-18",
    "capabilities": { "roots": { "listChanged": true }, "sampling": {} },
    "clientInfo": { "name": "ExampleClient", "version": "1.0.0" }
  }
}

// Response (server -> client)
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "protocolVersion": "2025-06-18",
    "capabilities": { "tools": { "listChanged": true } },
    "serverInfo": { "name": "ExampleServer", "version": "1.0.0" }
  }
}
```

And the `initialized` notification that follows -- note: no `id`:

```json
{ "jsonrpc": "2.0", "method": "notifications/initialized" }
```

## The three-phase lifecycle

1. **Initialization** -- the client **MUST** send `initialize` first; the
   server responds with its own capabilities; the client then sends the
   `initialized` notification. Neither side should send other requests
   before this handshake completes (pings are the one exception).
2. **Operation** -- normal request/response/notification traffic, limited
   to whatever capabilities both sides negotiated.
3. **Shutdown** -- no special MCP message; the transport itself signals
   termination (closing stdin for stdio, closing the HTTP connection for
   Streamable HTTP).

### Version negotiation

The client sends the latest protocol version it supports. If the server
also supports it, it echoes the same version back; otherwise it replies
with a version *it* supports, and the client disconnects if it can't accept
that either.

### Error handling

A failed `initialize` looks like an ordinary JSON-RPC error response:

```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "error": {
    "code": -32602,
    "message": "Unsupported protocol version",
    "data": { "supported": ["2024-11-05"] }
  }
}
```

In [ ]:
# See the handshake for real: examples/mcp_foundations/raw_jsonrpc_client.py
# speaks this exact sequence (initialize -> initialized -> tools/list ->
# tools/call) to examples/mcp_foundations/custom_server.py using nothing
# but the standard library -- no `mcp` package involved on the client side.
import subprocess
import sys
from pathlib import Path

script = str(Path.cwd().parent / "examples" / "mcp_foundations" / "raw_jsonrpc_client.py")
result = subprocess.run([sys.executable, script], capture_output=True, text=True, timeout=30)
print(result.stdout)

## Capability negotiation, and error codes

Capabilities establish which *optional* protocol features are available
for this session:

| Side | Capability | Means |
| --- | --- | --- |
| Client | `roots` | Can list filesystem roots for the server (notebook 27) |
| Client | `sampling` | Can fulfill `sampling/createMessage` requests (notebook 27) |
| Client | `elicitation` | Can fulfill mid-call user-input requests |
| Server | `tools` | Exposes callable tools (what Parts 1-2 focused on) |
| Server | `resources` | Exposes readable, URI-addressed data (notebook 26) |
| Server | `prompts` | Exposes reusable message templates (notebook 26) |
| Server | `logging` | Emits structured log messages |

Sub-flags like `listChanged` (list-change notifications) and `subscribe`
(per-item change notifications, resources only) refine what a declared
capability actually supports.

Standard JSON-RPC error codes you'll see in MCP responses:

| Code | Meaning |
| --- | --- |
| `-32601` | Method not found (for example, calling `roots/list` on a client that didn't declare `roots`) |
| `-32602` | Invalid params (for example, an unsupported protocol version) |
| `-32603` | Internal error |

MCP also defines protocol-specific codes for its own concepts -- for
example, `-32042` signals that a URL-mode elicitation must complete before
a request can proceed.